In [2]:
print("Hello, World!")
print("bye World!")
print("Goodbye, World!")

Hello, World!
bye World!
Goodbye, World!


In [1]:
import copernicusmarine
import xarray as xr
import numpy as np
from pathlib import Path 
import calendar

c:\Users\laiam\Desktop\Practicas FRESH-CARE\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_time_range(year, month):
    day_nbr = calendar.monthrange(year, month)[1]
    if len(str(month)) == 1:
        Month = "0" + str(month)
    else:
        Month = str(month)
    if len(str(day_nbr)) == 1:
        Day_nbr = "0" + str(day_nbr)
    else:
        Day_nbr = str(day_nbr)
    time_start = str(year)+ "-" + Month + "-01T00:00:00"
    time_end = str(year)+ "-" + Month + "-" + Day_nbr + "T23:59:59"
    month += 1
    if month > 12:
        month = 1
        year += 1

    return time_start, time_end, year, Month, month

In [3]:
def get_wind_data(time_start, time_end, year, Month):
  output_filename=f"wind_{year}-{Month}.nc"
  copernicusmarine.subset(
    dataset_id="cmems_obs-wind_glo_phy_my_l4_0.125deg_PT1H",
    variables=["eastward_wind", "northward_wind"],
    minimum_longitude=-179.9375,
    maximum_longitude=179.9375,
    minimum_latitude=66.56,
    maximum_latitude=90,
    start_datetime=time_start,
    end_datetime=time_end,
    output_filename=f"wind_{year}-{Month}.nc"
  )
  return output_filename

In [4]:
def wind24_data(year, Month, output_filename):
    ds = xr.open_dataset(output_filename)
    encoding = {"eastward_wind": {"dtype": "int16", "scale_factor": 0.01, "add_offset": 0,"_FillValue": -32767, "zlib": True, "complevel": 4},
                "northward_wind": {"dtype": "int16", "scale_factor": 0.01, "add_offset": 0,"_FillValue": -32767, "zlib": True, "complevel": 4}}
    ds_east24 = ds["eastward_wind"].resample(time="24h").mean()
    ds_north24 = ds["northward_wind"].resample(time="24h").mean()
    ws=xr.merge([ds_east24, ds_north24])
    filename=f"wind24_{year}-{Month}.nc"
    ws.to_netcdf(filename, encoding=encoding)
    ws.close()
    return filename

In [ ]:
for year in range(2010, 2016):
    for month in range(1,13):
        time_start, time_end, Month, year, month = get_time_range(year, month)
        raw = get_wind_data(time_start, time_end, year, Month)
        wind24_data(year, Month, raw)
        Path(raw).unlink()

INFO - 2026-10-20T10:34:51Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register
INFO - 2026-10-20T10:35:37Z - Selected dataset version: "202211"
INFO - 2026-10-20T10:35:37Z - Selected dataset part: "default"
WARNING - 2026-10-20T10:35:37Z - Some of your subset selection [66.56, 90.0] for the latitude dimension exceed the dataset coordinates [-89.9375, 89.9375]
100%|██████████| [00:48<00:00]
INFO - 2026-10-20T10:36:29Z - Total size of the download: 1.50 GB.


TypeError: '<=' not supported between instances of 'int' and 'str'